# Demonstration 3: a reversible bimolecular reaction in a one-body ramp potential

**What this validates:** the transport layer and the Fröhner–Noé acceptance rule,
with volume exclusion switched off.

**Analytic answer:** for a reversible reaction $A + B \rightleftharpoons C$ in a
static potential, the *local* equilibrium quotient follows the reaction's field work:

$$Q(x) \;=\; \frac{\langle n_C(x)\rangle}{\langle n_A n_B\rangle(x)}
  \;=\; \frac{k_F}{k_R}\, e^{-\Delta\phi(x)},
  \qquad \Delta\phi(x) = \sum_s \Delta\nu_s\,\phi_s(x)$$

Only $C$ is coupled to the field here ($\gamma_C = g$, $\gamma_A = \gamma_B = 0$), so
$\Delta\phi(x) = g\,\psi(x)$.

Note the denominator is $\langle n_A n_B\rangle$, the mean of the *product*, not the
product of the means. $A$ and $B$ are correlated through the conservation law, and only
the mean of the product satisfies the detailed-balance relation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vex_rddme import Simulation, Species, mu_ex_carnahan_starling
from vex_rddme.guards import suggest_tau
from vex_rddme import viz
from vex_rddme.observe import (
    Series, project, mu_ex_from_profile, align_additive_constant,
    relative_discrepancy, report_comparison, QuotientAccumulator,
)

## Setup

A 2D lattice with a linear ramp in $\psi$ along the short axis. Exclusion is off, so
this demonstration isolates transport and acceptance. If it fails, the fault is in one of those
two and not in the free-energy machinery.

In [ ]:
SHAPE      = (32, 24)     # (rows, field axis)
VOXEL_NM   = 20.0
GAMMA_C    = 1.5         # only C feels the field
K_F, K_R   = 60.0, 60.0
N_A = N_B  = 3000
TAU_S      = 2.0e-5      # exclusion off, so the bare CFL bound applies
N_STEPS    = 30_000
BURN_IN    = N_STEPS // 3
SAMPLE_EVERY = 25        # samples are correlated; space them out

ramp = np.arange(SHAPE[-1], dtype=float) / SHAPE[-1]
psi  = np.broadcast_to(ramp, SHAPE).copy()[None, ...]

sim = Simulation(
    shape=SHAPE, voxel_nm=VOXEL_NM,
    species=[
        Species("A", sigma_nm=0.0, gamma=np.array([0.0])),
        Species("B", sigma_nm=0.0, gamma=np.array([0.0])),
        Species("C", sigma_nm=0.0, gamma=np.array([GAMMA_C])),
    ],
    occupancy_cap=400, psi=psi, D_um2_s=1.0, tau_s=TAU_S,
    exclusion=False, seed=0,
)
sim.add_reaction("assoc", ["A", "B"], ["C"], K_F, K_R, typical_reactant_product=9.0)
sim.seed_uniform("A", N_A)
sim.seed_uniform("B", N_B)
sim.record_initial()
sim

## Run

Every guard is live during the run. If the timestep or the rate constants were wrong
for this configuration, the run would stop and say so rather than quietly producing a
plausible-looking wrong answer.

In [ ]:
acc = QuotientAccumulator(
    reactants=(0, 1), products=(2,), lattice=sim.lattice, axis=-1
)

for i in range(N_STEPS):
    sim.step()
    if i >= BURN_IN and (i - BURN_IN) % SAMPLE_EVERY == 0:
        acc.add(sim.state.counts)

sim.state.check_mass()          # exact: integers in, integers out
print(f"{acc.n} samples over {N_STEPS - BURN_IN} steps")
print("totals A, B, C:", sim.state.totals().tolist())
print("mean acceptance  forward %.4f   reverse %.4f"
      % (sim.reactions.mean_acceptance(0, 0), sim.reactions.mean_acceptance(0, 1)))

## Compare against the analytic prediction

Both curves are normalised by their means, so the comparison is of *shape*: the
absolute level is set by $k_F/k_R$, which this demonstration is not testing (that is the
well-mixed check in the test suite).

In [ ]:
Q_measured  = acc.quotient
Q_sem       = acc.sem
Q_predicted = (K_F / K_R) * np.exp(-GAMMA_C * ramp)

m = Q_measured  / Q_measured.mean()
p = Q_predicted / Q_predicted.mean()
s = Q_sem       / Q_measured.mean()

print(report_comparison("Q(x) shape vs (k_F/k_R) exp(-g psi(x))", m, p, sem=s))

# Exposed so the test suite can check this notebook's claim without a human reading it.
VERDICTS = {"Q(x) shape vs exp(-dPhi)": relative_discrepancy(m, p)["max"]}

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
viz.plot_profile(m, predicted=p, sem=s, ax=ax[0],
                 xlabel="voxel along the field axis", ylabel="Q(x) / mean",
                 label_measured="measured", label_predicted=r"$e^{-g\psi}$ (normalised)",
                 title="Demonstration 3: local equilibrium follows the field")
viz.show_lattice(sim.state.counts[2], sim.lattice, ax=ax[1], title="C occupancy")
plt.tight_layout(); plt.show()

## What to take from this

The measured quotient tracks $e^{-\Delta\phi(x)}$ across the box. Because $A$ and $B$
are uncoupled from the field, their densities stay flat. So the entire spatial
structure in $Q(x)$ comes from the acceptance rule, not from reactant transport.

**Try changing:**

- `GAMMA_C = 0`: the prediction becomes flat, and so should the measurement.
- `GAMMA_C = 4`: a steeper field. Watch for the `hop-probability-sum` guard: the
  Bernoulli factor grows for downhill moves and the timestep may no longer be small
  enough. That is the guard doing its job, not a bug.
- `K_F = 6.0`: a hundredfold slower reaction. The measurement will not have
  equilibrated in `N_STEPS`, and the discrepancy will grow. Reaction equilibration time
  scales as $1/k$.